In [0]:
-- ══════════════════════════════════════════════════════════════════════════════
-- DiD v6 — cohort dedup fix
-- Members with multiple sends: if opened ANY, they're an opener
-- ══════════════════════════════════════════════════════════════════════════════

WITH
sent AS (
    SELECT DISTINCT f.message_profile_id, e.CDE_ID
    FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback f
    LEFT JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
           ON e.Email_Address = f.Email_Address
    WHERE f.journey_id = '96b494ce-f57e-4068-b24c-049fd5684fee'
      AND f.journey_action_id IN (
            'ef8134cd-3dd9-4553-b8cf-c273db9d84b9',
            'c6ba2ed4-5814-417d-8d46-281932defbc5')
      AND f.feedback_status = 'sent'
      AND e.CDE_ID IS NOT NULL
),
opens AS (
    SELECT DISTINCT message_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_adobe_email_tracking
    WHERE interaction_type = 'open'
      AND CAST(timestamp AS DATE) <= DATE '2026-08-26'
),
cohorts AS (
    SELECT
        s.CDE_ID,
        CASE WHEN MAX(CASE WHEN o.message_profile_id IS NOT NULL THEN 1 ELSE 0 END) = 1
             THEN 'opener' ELSE 'non_opener' END AS cohort
    FROM sent s
    LEFT JOIN opens o USING (message_profile_id)
    GROUP BY s.CDE_ID
),

mm_txn_keys AS (
    SELECT DISTINCT t.CDE_ID, t.Transaction_Number, t.LocationId
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_film fm ON fm.film_id = t.film_id
    WHERE t.SCENEFLAG  = 'Yes'
      AND t.TicketFLAG = 'Yes'
      AND t.CDE_ID IS NOT NULL
      AND t.VisitDateId BETWEEN 20260702 AND 20260821
      AND fm.title_id  = '38680'
),
ineligible_members AS (
    SELECT DISTINCT k.CDE_ID
    FROM mm_txn_keys k
    JOIN cpx_dataanalytics_gold.customer360.c360_f_transaction t
      ON t.Transaction_Number = k.Transaction_Number
     AND t.CDE_ID              = k.CDE_ID
     AND t.LocationId          = k.LocationId
    WHERE t.SCENE_Points_Redeemed > 0
),

trans_slim AS (
    SELECT
        t.CDE_ID, c.cohort, dt.datefull,
        loc.LC_Location_Type_Description  AS loc_type,
        loc.LC_Trade_Brand_Description    AS brand,
        fm.title_id, t.TicketFLAG, t.LocationId, t.Net_Revenue,
        CONCAT_WS('|', CAST(t.VisitDateId AS STRING),
                       CAST(t.LocationId  AS STRING),
                       t.CDE_ID) AS visit_key
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cohorts c ON c.CDE_ID = t.CDE_ID
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date dt ON dt.dateid = t.VisitDateId
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON loc.LocationID = t.LocationId
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film fm ON fm.film_id = t.film_id
    WHERE t.SCENEFLAG = 'Yes'
      AND dt.datefull BETWEEN DATE '2026-01-01' AND DATE '2026-09-11'
      AND (
            loc.LC_Location_Type_Description = 'Cineplex Theatre'
         OR (loc.LC_Location_Type_Description = 'Restaurant/Store/Games'
             AND loc.LC_Trade_Brand_Description IN ('Rec Room','Playdium'))
          )
),

cinema_tickets AS (
    SELECT DISTINCT
        ts.CDE_ID, ts.cohort, ts.datefull AS cinema_date,
        CASE
            WHEN ts.title_id  = '38680'
                 AND ts.datefull BETWEEN DATE '2026-07-02' AND DATE '2026-08-05'
                 AND ts.cohort = 'opener'
                 AND ie.CDE_ID IS NULL                       THEN 'A'
            WHEN ts.title_id  = '38680'
                 AND ts.datefull BETWEEN DATE '2026-07-02' AND DATE '2026-08-05'
                 AND ts.cohort = 'non_opener'
                 AND ie.CDE_ID IS NULL                       THEN 'B'
            WHEN ts.title_id  = '38680'
                 AND ts.datefull BETWEEN DATE '2026-08-06' AND DATE '2026-08-21'
                 AND ts.cohort = 'opener'
                 AND ie.CDE_ID IS NULL                       THEN 'A_c'
            WHEN ts.title_id  = '38680'
                 AND ts.datefull BETWEEN DATE '2026-08-06' AND DATE '2026-08-21'
                 AND ts.cohort = 'non_opener'
                 AND ie.CDE_ID IS NULL                       THEN 'B_c'
            WHEN ts.title_id <> '38680'
                 AND ts.datefull BETWEEN DATE '2026-01-01' AND DATE '2026-06-30'
                 AND ts.cohort = 'opener'                    THEN 'C'
            WHEN ts.title_id <> '38680'
                 AND ts.datefull BETWEEN DATE '2026-01-01' AND DATE '2026-06-30'
                 AND ts.cohort = 'non_opener'                THEN 'D'
        END AS cell_id
    FROM trans_slim ts
    LEFT JOIN ineligible_members ie ON ie.CDE_ID = ts.CDE_ID
    WHERE ts.loc_type   = 'Cineplex Theatre'
      AND ts.TicketFLAG = 'Yes'
),
cinema_cells AS (SELECT * FROM cinema_tickets WHERE cell_id IS NOT NULL),

lbe_visits AS (
    SELECT CDE_ID, datefull AS lbe_date,
           visit_key AS lbe_visit_key,
           SUM(Net_Revenue) AS lbe_revenue
    FROM trans_slim
    WHERE loc_type = 'Restaurant/Store/Games'
    GROUP BY CDE_ID, datefull, visit_key
),

ticket_attribution AS (
    SELECT /*+ RANGE_JOIN(l, 21) */
        c.CDE_ID, c.cell_id, c.cinema_date,
        l.lbe_date, l.lbe_visit_key, l.lbe_revenue,
        ROW_NUMBER() OVER (PARTITION BY c.CDE_ID, c.cell_id, c.cinema_date
                           ORDER BY l.lbe_date NULLS LAST) AS lbe_rank
    FROM cinema_cells c
    LEFT JOIN lbe_visits l
      ON  l.CDE_ID   = c.CDE_ID
     AND  l.lbe_date >= c.cinema_date
     AND  l.lbe_date <= DATE_ADD(c.cinema_date, 21)
),
first_lbe_per_ticket AS (
    SELECT CDE_ID, cell_id, cinema_date, lbe_date, lbe_visit_key, lbe_revenue
    FROM ticket_attribution
    WHERE lbe_rank = 1
),

member_stats AS (
    SELECT cell_id, CDE_ID,
           COUNT(*) AS total_tickets,
           COUNT(lbe_date) AS qualifying_tickets,
           LEAST(COUNT(lbe_date), 10) AS payout_events
    FROM first_lbe_per_ticket
    GROUP BY cell_id, CDE_ID
),
member_revenue AS (
    SELECT cell_id, CDE_ID, SUM(lbe_revenue) AS attributed_revenue
    FROM (
        SELECT DISTINCT cell_id, CDE_ID, lbe_visit_key, lbe_revenue
        FROM first_lbe_per_ticket
        WHERE lbe_visit_key IS NOT NULL
    )
    GROUP BY cell_id, CDE_ID
),

cell_stats AS (
    SELECT ms.cell_id,
           COUNT(DISTINCT ms.CDE_ID) AS total_viewers,
           SUM(CASE WHEN ms.qualifying_tickets > 0 THEN 1 ELSE 0 END) AS cross_visitors,
           SUM(ms.payout_events) AS payout_events,
           COALESCE(SUM(mr.attributed_revenue), 0) AS attributed_revenue
    FROM member_stats ms
    LEFT JOIN member_revenue mr USING (cell_id, CDE_ID)
    GROUP BY ms.cell_id
),
cell_rates AS (
    SELECT cell_id,
        CASE cell_id
            WHEN 'A'   THEN 'Openers × M&M (CLEAN: Jul 2 – Aug 5)'
            WHEN 'B'   THEN 'Non-openers × M&M (CLEAN: Jul 2 – Aug 5)'
            WHEN 'A_c' THEN 'Openers × M&M (CENSORED: Aug 6 – Aug 21)'
            WHEN 'B_c' THEN 'Non-openers × M&M (CENSORED: Aug 6 – Aug 21)'
            WHEN 'C'   THEN 'Openers × Other movies (12mo hist)'
            WHEN 'D'   THEN 'Non-openers × Other movies (12mo hist)'
        END AS description,
        total_viewers, cross_visitors, payout_events, attributed_revenue,
        cross_visitors * 1.0 / total_viewers AS visit_rate,
        attributed_revenue    / total_viewers AS rev_per_viewer
    FROM cell_stats
),
did_summary AS (
    SELECT
        MAX(CASE WHEN cell_id='A' THEN visit_rate     END) AS vr_A,
        MAX(CASE WHEN cell_id='B' THEN visit_rate     END) AS vr_B,
        MAX(CASE WHEN cell_id='C' THEN visit_rate     END) AS vr_C,
        MAX(CASE WHEN cell_id='D' THEN visit_rate     END) AS vr_D,
        MAX(CASE WHEN cell_id='A' THEN rev_per_viewer END) AS rv_A,
        MAX(CASE WHEN cell_id='B' THEN rev_per_viewer END) AS rv_B,
        MAX(CASE WHEN cell_id='C' THEN rev_per_viewer END) AS rv_C,
        MAX(CASE WHEN cell_id='D' THEN rev_per_viewer END) AS rv_D,
        MAX(CASE WHEN cell_id='A' THEN total_viewers  END) AS n_A,
        SUM(CASE WHEN cell_id IN ('A','B','A_c','B_c') THEN payout_events ELSE 0 END) AS total_payouts,
        SUM(CASE WHEN cell_id IN ('A','B','A_c','B_c') THEN attributed_revenue ELSE 0 END) AS total_offer_revenue
    FROM cell_rates
)

SELECT
    cell_id AS cell, description, total_viewers, cross_visitors,
    ROUND(visit_rate * 100, 3) AS visit_rate_pct,
    payout_events, ROUND(attributed_revenue, 2) AS attributed_revenue,
    ROUND(rev_per_viewer, 2) AS rev_per_viewer,
    CAST(NULL AS DOUBLE) AS did_visit_pp,
    CAST(NULL AS DOUBLE) AS did_rev_per_viewer,
    CAST(NULL AS BIGINT) AS total_payouts,
    CAST(NULL AS DOUBLE) AS cost_dollars,
    CAST(NULL AS DOUBLE) AS total_offer_revenue,
    CAST(NULL AS DOUBLE) AS incr_revenue_dollars,
    CAST(NULL AS DOUBLE) AS net_dollars
FROM cell_rates
UNION ALL
SELECT
    'DiD', 'Effect estimate + total revenue exposure',
    CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS DOUBLE),
    CAST(NULL AS BIGINT), CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE),
    ROUND(((vr_A - vr_B) - (vr_C - vr_D)) * 100, 3),
    ROUND((rv_A - rv_B) - (rv_C - rv_D), 2),
    total_payouts,
    ROUND(total_payouts * 250 * 0.009237, 2),
    ROUND(total_offer_revenue, 2),
    ROUND(((rv_A - rv_B) - (rv_C - rv_D)) * n_A, 2),
    ROUND(((rv_A - rv_B) - (rv_C - rv_D)) * n_A
        - total_payouts * 250 * 0.009237, 2)
FROM did_summary
ORDER BY CASE cell WHEN 'A' THEN 1 WHEN 'B' THEN 2
                   WHEN 'A_c' THEN 3 WHEN 'B_c' THEN 4
                   WHEN 'C' THEN 5 WHEN 'D' THEN 6 ELSE 7 END;